# Занятие 3. Основы Python

In [ ]:
# 📥 Установка пакетов и библиотек
!pip install numpy
!pip install pandas
!pip install matplotlib
!pip install scipy
!pip install rdkit
!pip install plotly

In [ ]:
# 🛠️ Загрузка необходимых пакетов в среду выполнения
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import plotly.graph_objects as go

## 1. 🔬 Глава 1. Модуль scipy

**Scipy** - библиотека Python, предназначенная для выполнения научных и инженерных расчётов.
Библиотека работает с многомерными массивами типа **ndarray** из библиотеки NumPy. 

Подмодули (1):
| Название | Описание | 
|---|---|
| `scipy.interpolate` | интерполяция |
| `scipy.optimize` | корни уравнений, минимизация, подгонка |
| `scipy.integrate` | интегралы и дифференциальные уравнения |
| `scipy.stats` | статистика |

Подмодули (2):
| Название | Описание | 
|---|---|
| `scipy.cluster` | методы кластеризации |
| `scipy.constants` | физические константы (R, N_A, h, c, ...) |
| `scipy.fft` | преобразование Фурье и обработка сигналов и спектров |
| `scipy.linalg` | методы линейной алгебры |
| `scipy.ndimage` | обработка изображений |
| `scipy.odr` | регрессия и методы ортогональных расстояний |
| `scipy.signal` | обработка сигналов: фильтрация, сглаживание, поиск пиков |
| `scipy.sparse` | работа с разреженными массивами |
| `scipy.spatial` | вычислительная геометрия |
| `scipy.special` | специальные функции |

### ∫🧐 1.1. Интегральные функции

In [ ]:
# Импорт функций подмодуля в среду выполнения
from scipy import integrate

In [ ]:
# В модуле scipy.integrate есть несколько функций для вычисления определённых интегралов. Наиболее простой является функция quad(),
# которая вычисляет обычный интеграл от одной переменной
# Проинтегрируем функцию sin(x) в диапазоне от 0 до pi

def f(x):
    return np.sin(x)

a, b = [-np.pi, np.pi] # integral range

I, err = integrate.quad(f, a, b)
print(I, err)

# Визуализация
x = np.linspace(a - 0.5, b + 0.5, 300)
x_fill = np.linspace(a, b, 200)

fig, ax = plt.subplots(figsize=(11, 4.5))
ax.plot(x, f(x), color='#228CDB', label='f(x) = sin(x)')
ax.fill_between(x_fill, f(x_fill), alpha=0.3, color='#228CDB', label=f'Площадь = {I:.6f}')
ax.axhline(0, color='black', linewidth=1)
ax.set_xlabel('x')
ax.set_ylabel('f(x)')
ax.legend()
ax.grid(True)
plt.show()

print(f"Численно:    {I:.10f}")
print(f"Погрешность: {err:.1e}")

In [ ]:
# Решение ОДУ с использованием функции solve_ivp() 
# dy/dx = -2y

def dydx(x, y):            # правая часть ОДУ: возвращает производную
    return -2.0 * y

y0 = 1.0
x_eval = np.linspace(0, 3, 50)                 # точки, где хотим знать решение
sol = integrate.solve_ivp(dydx, (0, 3), [y0], t_eval=x_eval)
print(sol)

fig, ax = plt.subplots(figsize=(11, 4.5))
ax.plot(x_eval, np.exp(-2.0 * x_eval), '--', color='gray', label='Аналитически: y = e$^{-2x}$')
ax.plot(sol.t, sol.y[0], 'o', color='#228CDB', markersize=4, label='Численно: solve_ivp')

ax.set_xlabel('x')
ax.set_ylabel('y')
ax.legend()
ax.grid(True)
plt.show()

print(f"Максимальное расхождение с аналитикой: "
      f"{np.abs(sol.y[0] - np.exp(-2.0 * sol.t)).max():.1e}")


In [ ]:
# Система ОДУ
# Уравнение второго порядка (гармонический осциллятор) 
# y'' = −y стандартным приёмом сводится к системе
# двух уравнений первого порядка - вводим вторую переменную y2 = y':
#   dy1/dx = y2
#   dy2/dx = −y1
# Аналитическое решение при y1(0)=1, y2(0)=0:  y1 = cos x,  y2 = −sin x

def rhs(x, y):
    y1, y2 = y                  # y - вектор из двух переменных
    return [y2, -y1]            # возвращаем вектор производных

x_eval = np.linspace(0, 10, 200)
sol = integrate.solve_ivp(rhs, (0, 10), [1.0, 0.0], t_eval=x_eval, rtol=1e-6)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4.5), width_ratios=[2, 1])

# Слева: решения от x
ax1.plot(sol.t, np.cos(sol.t), '--', color='gray', label='Аналитически: cos x')
ax1.plot(sol.t, sol.y[0], color='#228CDB', label='y₁ (решение)')
ax1.plot(sol.t, sol.y[1], color='#E76F51', label='y₂ (производная)')
ax1.set_xlabel('x')
ax1.set_ylabel('y')
ax1.legend()
ax1.grid(True)

# Справа: фазовый портрет — y₂ против y₁
ax2.plot(sol.y[0], sol.y[1], color='#228CDB')
ax2.plot(1, 0, 'ko', label='старт')
ax2.set_xlabel('y₁')
ax2.set_ylabel('y₂')
ax2.set_aspect('equal')
ax2.set_title('Фазовый портрет')
ax2.legend()
ax2.grid(True)

plt.tight_layout()
plt.show()

print(f"Максимальная ошибка: {np.abs(sol.y[0] - np.cos(sol.t)).max():.1e}")

### 📈 1.2. Интерполяция

Интерполяция - восстановление значений функции между известными точками данных. Это полезно, когда экспериментальные измерения проведены в ограниченном числе точек. 

In [ ]:
from scipy import interpolate

In [ ]:
# Линейная и кубическая интерполяция
# Одномерная интерполяция scipy.interpolate.interp1d
# Одномерная кубическая интерполяция scipy.interpolate.CubicSpline

def f(x):
    return np.sin(x)

n = np.linspace(0, 2*np.pi, 8)
x = np.linspace(0, 2*np.pi, 100)
x_ext = np.linspace(0, 4*np.pi, 200)

y_meas = f(n)
y_real = f(x)

linear_interpolation = interpolate.interp1d(n, y_meas, kind='linear')
y_interp1 = linear_interpolation(x)

cubic_interpolation = interpolate.interp1d(n, y_meas, kind='cubic')
                      # linear     - отрезки прямых между соседними точками (по умолчанию)
                      # nearest    - значение ближайшей точки; посередине между точками берётся левая
                      # nearest-up - то же, но посередине между точками берётся правая
                      # zero       - сплайн нулевого порядка: ступенчатая (кусочно-постоянная) функция
                      # slinear    - сплайн первого порядка (результат совпадает с 'linear')
                      # quadratic  - квадратичный сплайн (гладкая кривая из парабол)
                      # cubic      - кубический сплайн (гладкая кривая из полиномов 3-й степени)
                      # previous   - «удержание» значения предыдущей точки (ступенька)
                      # next       - значение следующей точки
y_interp2 = cubic_interpolation(x)

# Периодическая интерполяция: значения в начале и конце диапазона должны совпадать (y[0] == y[-1]).
# Такой сплайн гладко сшивается на границе и продолжается за пределы данных
periodic_interpolation = interpolate.CubicSpline(n, y_meas, bc_type='periodic')
y_interp3 = periodic_interpolation(x_ext)

fig, ax = plt.subplots(figsize=(11, 4.5))
ax.plot(n, y_meas, 'bs', label='Сырые данные')
ax.plot(x, y_real, 'orange', label='Реальная функция')
ax.plot(x, y_interp1, 'r', label='Линейная интерполяция')
ax.plot(x, y_interp2, 'g', label='Кубическая интерполяция')
ax.plot(x_ext, y_interp3, 'm--', label='Периодический сплайн')
ax.grid()
ax.legend()
plt.show()

In [ ]:
# Интерполяция спектра поглощения
# «Эксперимент»: спектр - сумма трёх гауссовых полос поглощения

def gauss(wl, A, center, width):
    return A * np.exp(-(wl - center)**2 / (2 * width**2))

def spectrum(wl):
    return (gauss(wl, 0.40, 230, 20) + 
            gauss(wl, 0.25, 300, 15) +
            gauss(wl, 0.30, 250, 10))

# Измерения сделаны в ограниченном числе точек
wavelength = np.linspace(200, 400, 15)    # нм
absorbance = spectrum(wavelength)

# Тонкая сетка для интерполяции
wl_fine = np.linspace(200, 400, 200)
spline = interpolate.CubicSpline(wavelength, absorbance)

fig, ax = plt.subplots(figsize=(11, 4.5))
ax.plot(wl_fine, spectrum(wl_fine), '--', label='Истинный спектр', color='#2D3142')
ax.plot(wavelength, absorbance, 'o', label='Эксперимент', color='#228CDB')
ax.plot(wl_fine, spline(wl_fine), '-', label='Кубический сплайн', color="#E7242D")
ax.set_xlabel('Длина волны, нм')
ax.set_ylabel('Оптическая плотность')
ax.set_xlim([200, 400])
ax.set_ylim([0, np.max(absorbance)+0.1])
ax.legend()
ax.grid(True)
plt.show()

In [ ]:
# Двумерная интерполяция: поверхность потенциальной энергии (ППЭ)
# scipy.interpolate.RegularGridInterpolator((x,y), z)

# Скан энергии по двум торсионным углам молекулы с шагом 30°, кДж/моль
def E(phi1, phi2):
    return (2.0 * (1 + np.cos(np.radians(3*phi1)))
          + 1.5 * (1 + np.cos(np.radians(3*phi2)))
          + 1.5 * np.cos(np.radians(phi1 - phi2)))

phi = np.arange(-180, 181, 30)
P1, P2 = np.meshgrid(phi, phi, indexing='ij')
E_scan = E(P1, P2)

# Интерполятор по регулярной сетке: оси + матрица значений
interp = interpolate.RegularGridInterpolator((phi, phi), E_scan, method='cubic')

# Вызывается на массиве точек (φ1, φ2), поэтому stack по последней оси
phi_fine = np.linspace(-180, 180, 181)
P1f, P2f = np.meshgrid(phi_fine, phi_fine, indexing='ij')
E_interp = interp(np.stack([P1f, P2f], axis=-1))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4.5), sharey=True)

ax1.pcolormesh(P1, P2, E_scan, shading='nearest', vmin=E_interp.min(), vmax=E_interp.max(), cmap='RdYlBu_r')
ax1.set_title('Скан с шагом 30°')
ax1.set_xlabel('φ1')
ax1.set_ylabel('φ2')

im = ax2.contourf(P1f, P2f, E_interp, levels=20, cmap='RdYlBu_r')
ax2.set_title('Интерполированная ППЭ')
ax2.set_xlabel('φ1')

ticks = np.arange(-180, 181, 60)
for ax in (ax1, ax2):
    ax.set_xticks(ticks)
    ax.set_xlim(-180, 180)
    ax.set_ylim(-180, 180)
    ax.set_aspect('equal')
ax1.set_yticks(ticks)

fig.colorbar(im, ax=(ax1, ax2), label='E, кДж/моль')
plt.show()

In [ ]:
# Опасность экстраполяции
# Измерили константу скорости реакции при 300-350 К, хотим оценить константу скорости при 250 К.
R, Ea, lnA = 8.314, 60e3, 23.0    # Дж/(моль·К), Дж/моль, ln(с⁻¹)

def lnk_true(T):
    return lnA - Ea / (R * T)

# Эксперимент в узком интервале температур + шум измерений
T_exp = np.linspace(350, 300, 6)   # от большей T к меньшей, чтобы 1/T возрастала
np.random.seed(42)
lnk_exp = lnk_true(T_exp) + np.random.normal(0, 0.05, len(T_exp))
x_exp = 1000 / T_exp

# Сплайн проходит точно через точки; прямая - физическая модель
spline = interpolate.CubicSpline(x_exp, lnk_exp)
a, b = np.polyfit(x_exp, lnk_exp, 1)

T_all = np.linspace(250, 400, 300)
x_all = 1000 / T_all

fig, ax = plt.subplots(figsize=(11, 4.5))
ax.plot(x_exp, lnk_exp, 'o', color='#228CDB', label='Эксперимент (300–350 К)')
ax.plot(x_all, a * x_all + b, 'g', label='Физическая модель')
ax.plot(x_all, spline(x_all), 'r', label='Кубический сплайн')
ax.set_xlabel('1000/T, К$^{-1}$')
ax.set_ylabel('ln k')
ax.set_ylim(-4, 6)
ax.set_xlim(2.6, 3.6)
ax.legend()
ax.grid(True)
plt.show()

In [ ]:
# Явление Рунге. Почему бы не провести через все точки один полином высокой степени?

def lorentz(x):
    return 1 / (1 + 25 * x**2)

x_exp = np.linspace(-1, 1, 11)
y_exp = lorentz(x_exp)

poly   = interpolate.lagrange(x_exp, y_exp)    # полином Лагранжа 10-й степени
spline = interpolate.CubicSpline(x_exp, y_exp)

x_fine = np.linspace(-1, 1, 400)

fig, ax = plt.subplots(figsize=(11, 4.5))
ax.plot(x_fine, lorentz(x_fine), '--', color='gray', label='Истинный контур')
ax.plot(x_exp, y_exp, 'o', color='#228CDB', label='Точки измерений')
ax.plot(x_fine, poly(x_fine), 'r', label='Полином 10-й степени')
ax.plot(x_fine, spline(x_fine), 'g', label='Кубический сплайн')
ax.legend()
ax.grid(True)
plt.show()

### 📐 1.3. Поиск корня и минимума функций.

Применение подмодуля `scipy.optimize` сводится к двум базовым задачам:

1. **найти корень**: $f(x) = 0$;
2. **найти минимум**: $\min f(x)$.

Поскольку задачи взаимно превращаются друг в друга, методы внутри подмодуля `optimize` похожи: почти все они **итерационные** - стартуют с начального
приближения и шаг за шагом улучшают его.

| Функция | Задача | Что нужно задать |
|---|---|---|
| `brentq(f, a, b)` | корень одного уравнения | отрезок $[a, b]$, где $f$ меняет знак |
| `newton(f, x0)` | корень одного уравнения | стартовую точку $x_0$ |
| `minimize(f, x0)` | минимум функции одной или многих переменных | стартовую точку |
| `fsolve(f, x0)` | корень системы уравнений | вектор стартовых точек |

In [ ]:
from scipy import optimize

In [ ]:
# Трансцендентное уравнение - уравнение без точного аналитического решения в элементарных функциях, 
# корень которого можно найти только численно:

# Решим уравнение sin(x) = x/2. Переносим всё в левую часть: f(x) = sin(x) - x/2
# Корень уравнения функции - f(x) = 0 

def f(x):
    return np.sin(x) - x/2

# scipy.optimize.brentq ищет корень на отрезке [a, b], где f(a) и f(b) разных знаков

a, b = -3, -1
root = optimize.brentq(f, a, b)

# Визуализация
fig, ax = plt.subplots(figsize=(11, 4.5))
x = np.linspace(-3, 3, 300)
ax.plot(x, f(x), label='f(x) = sin(x) − x/2')
ax.axhline(0, color='black', linewidth=1)

ax.plot(root, 0, 'r*', markersize=14, label=f'Корень: x = {root:.4f}')
ax.axvspan(a, b, color='lightgray', alpha=0.5, label=f'Отрезок поиска [{a}, {b}]')
ax.set_xlabel('x')
ax.set_ylabel('f(x)')
ax.legend()
ax.grid(True)
plt.show()

In [ ]:
# Поиск минимума как поиск корня производной
# В точке минимума гладкой функции её производная равна нулю.
# Найдём минимум той же f(x) = sin(x) − x/2, решив уравнение f'(x) = 0.

from scipy.differentiate import derivative

def f(x):
    return np.sin(x) - x/2

def df(x):
    # Численная производная: scipy сам подбирает шаг конечных разностей
    return derivative(f, x).df

# Корень производной ищем знакомым brentq: на [−2, 0] она меняет знак
x_min = optimize.brentq(df, -2, 0)

# Визуализация: функция и её производная с общей осью x
x = np.linspace(-3, 3, 400)
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(11, 7), sharex=True)

ax1.plot(x, f(x), color='#228CDB', label='f(x) = sin x − x/2')
ax1.plot(x_min, f(x_min), 'r*', markersize=16, label=f'Минимум: x = {x_min:.4f}')
ax1.set_ylabel('f(x)')
ax1.legend()
ax1.grid(True)

ax2.plot(x, df(x), color='#E76F51', label="f'(x) численно (scipy.differentiate)")
ax2.axhline(0, color='black', linewidth=1)
ax2.plot(x_min, 0, 'r*', markersize=14, label=f'Корень производной: x = {x_min:.4f}')
ax2.set_xlabel('x')
ax2.set_ylabel("f'(x)")
ax2.legend()
ax2.grid(True)

plt.tight_layout()
plt.show()

print(f"Корень f'(x) = 0:  x = {x_min:.6f}   (точно: −π/3 = {-np.pi/3:.6f})")
print(f"Сверка через minimize_scalar: "
      f"x = {optimize.minimize_scalar(f, bounds=(-3, 0), method='bounded').x:.6f}")


In [ ]:
# Итерационный поиск корня от стартовой точки
# То же уравнение sin(x) = x/2. Метод Ньютона: в текущей точке проводим
# касательную и шагаем туда, где она пересекает ноль.
# В scipy это scipy.optimize.newton(f, x0, fprime)

def f(x):
    return np.sin(x) - x/2

# Обёртка запоминает точки, в которых scipy вычисляет f,- итерации
trajectory = []
def f_traced(x):
    trajectory.append(float(x))
    return f(x)

x0 = -2.9   # стартовая точка
root, info = optimize.newton(f_traced, x0,
                                   fprime=lambda x: np.cos(x) - 0.5,
                                   full_output=True)
trajectory = np.array(trajectory)

# Визуализация
fig, ax = plt.subplots(figsize=(11, 4.5))
x = np.linspace(-3, 3, 300)
ax.plot(x, f(x), label='f(x) = sin(x) - x/2')
ax.axhline(0, color='black', linewidth=1)

ax.plot(trajectory, f(trajectory), 'o--', color='#E76F51', label=f'Итерации из x₀ = {x0}')
ax.plot(root, 0, 'r*', markersize=14, label=f'Корень: x = {root:.4f}')
ax.set_xlabel('x')
ax.set_ylabel('f(x)')
ax.legend()
ax.grid(True)
plt.show()

print(f"Сошлось за {info.iterations} итераций")
for i, xi in enumerate(trajectory):
    print(f"{i:8d} | {xi:10.6f} | {f(xi):10.2e}")

In [ ]:
# Корень системы уравнений: fsolve
# brentq и newton работают только с одним уравнением. Для системы
# используется fsolve(f, x0)

# Найдём точки пересечения окружности и гиперболы:
#   x^2 + y^2 = 4
#   x*y = 1

def equations(v):
    x, y = v
    return [x**2 + y**2 - 4,   # окружность
            x*y - 1]           # гипербола

# У системы четыре корня - какой найдётся, зависит от стартовой точки
starts = [(2, 0.3), (0.3, 2), (-2, -0.3), (-0.3, -2)]
roots = [optimize.fsolve(equations, x0) for x0 in starts]

for x0, r in zip(starts, roots):
    print(f"старт {x0} -> корень ({r[0]: .4f}, {r[1]: .4f})")

# Визуализация: обе кривые, старты (крестики) и найденные корни (точки)
theta = np.linspace(0, 2*np.pi, 200)
t = np.linspace(0.15, 6, 200)

fig, ax = plt.subplots(figsize=(5.5, 5.5))
ax.plot(2*np.cos(theta), 2*np.sin(theta), label="$x^2+y^2=4$")
ax.plot(t, 1/t, color="tab:orange", label="$xy=1$")
ax.plot(-t, -1/t, color="tab:orange")
for x0, r in zip(starts, roots):
    ax.plot(*r, "ko")
    ax.plot(*x0, "x", color="gray")
ax.set_xlim(-3, 3); ax.set_ylim(-3, 3)
ax.set_aspect("equal")
ax.set_xlabel("x"); ax.set_ylabel("y")
ax.legend(loc="upper left")
ax.set_title("fsolve: 4 старта - 4 разных корня")
plt.tight_layout()
plt.show()

In [ ]:
# Поиск корня как поиск минимума функции ошибки
# E(x) = (sin(x) - x/2)²
# E(x) обращается в ноль ровно в корнях уравнения, поэтому чтобы
# найти корень нужно найти минимум функции E. Минимизатор стартует из точки x0
# и итерационно спускается «вниз по склону», пока ошибка не перестанет уменьшаться

def f(x):
    return np.sin(x) - x/2

def E(x):
    return f(x)**2

x0 = -2.9                # стартовая точка
trajectory = [x0]        # callback запоминает точку после каждой итерации
res = optimize.minimize(E, x0, callback=lambda xk: trajectory.append(float(xk[0])))
trajectory = np.array(trajectory)
root = res.x[0]

# Визуализация: исходная функция и ландшафт ошибки с общей осью x
x = np.linspace(-3, 3, 400)
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(11, 7), sharex=True)

ax1.plot(x, f(x), color='#228CDB', label='f(x) = sin x − x/2')
ax1.axhline(0, color='black', linewidth=1)
ax1.plot(root, 0, 'r*', markersize=14, label=f'Корень: x = {root:.4f}')
ax1.set_ylabel('f(x)')
ax1.legend()
ax1.grid(True)

ax2.plot(x, E(x), 'k', label='E(x) = f(x)²')
ax2.plot(trajectory, E(trajectory), 'o--', color='#E76F51', label=f'Итерации из x₀ = {x0}')
ax2.plot(root, E(root), 'r*', markersize=16, label=f'Минимум: x = {root:.4f}')
ax2.set_xlabel('x')
ax2.set_ylabel('E(x)')
ax2.legend()
ax2.grid(True)

plt.tight_layout()
plt.show()

print("Итерация |     x      |    E(x)")
for i, xi in enumerate(trajectory):
    print(f"{i:8d} | {xi:10.6f} | {E(xi):10.2e}")

### 📏 1.4. Методы аппроксимации

**Функция потерь (Loss function)** - это функция, которая отображает значения одной или нескольких переменных на действительное число, характеризующее согласие модели и данных.

Пусть модель с параметрами $\theta$ предсказывает $y_\text{модель}(x_i;\,\theta)$,
а измерено $y_i$. Разность

$$r_i = y_i - y_\text{модель}(x_i;\,\theta)$$

называется **невязкой** (остатком). Из невязок собирают одно число - ошибку модели:

**RMSE** = $\sqrt{\dfrac{1}{N}\sum_i r_i^2}$, выраженную в **единицах измеряемой величины**

**Подгонка параметров** (фитинг) — это минимизация RMSE по параметрам:
тот же итерационный поиск минимума, что и в разделе 4, только теперь
$f(x) = \mathrm{RMSE}$, а $x$ - параметры модели $\theta$.

In [ ]:
# Функция потерь
np.random.seed(42)
a_true, b_true = 2.5, 1.0
x_data = np.linspace(0, 10, 20)
noise = np.random.normal(0, 1.0, size=x_data.size)
y_data = a_true * x_data + b_true + noise

# Модель задаётся обычной функцией: первый аргумент - x, дальше - параметры
def line(x, a, b):
    return a * x + b

popt, pcov = optimize.curve_fit(line, x_data, y_data)
a_fit, b_fit = popt

print(f"a = {a_fit:.3f} (истина {a_true})")
print(f"b = {b_fit:.3f} (истина {b_true})")

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))

# Слева: данные и подогнанная прямая
ax1.plot(x_data, y_data, 'bo', label='Данные с шумом')
ax1.plot(x_data, line(x_data, *popt), 'r-', label='curve_fit')   # * распаковывает popt
ax1.set_xlabel('x')
ax1.set_ylabel('y')
ax1.legend()
ax1.grid(True)

# Справа: ландшафт функции ошибки RMSE(a, b), который минимизировал curve_fit
a_values = np.linspace(1.5, 3.5, 100)
b_values = np.linspace(-3, 5, 100)
rmse_grid = np.zeros((len(b_values), len(a_values)))
for i, b in enumerate(b_values):
    for j, a in enumerate(a_values):
        rmse_grid[i, j] = np.sqrt(np.mean((y_data - line(x_data, a, b))**2))

cs = ax2.contourf(a_values, b_values, rmse_grid, levels=30, cmap='RdYlBu_r')
fig.colorbar(cs, ax=ax2, label='RMSE')
ax2.plot(a_fit, b_fit, 'ro', markersize=15, label='Минимум (curve_fit)')
ax2.set_xlabel('a')
ax2.set_ylabel('b')
ax2.legend()
ax2.grid(True)

plt.tight_layout()
plt.show()

In [ ]:
# Выбор начальных условий

def U(x, y):
    return (+ 2.5 * np.exp(-(x**2 + y**2) / 1.5)                  # центральный холм
            - 2.0 * np.exp(-(x**2 + (y - 3)**2) / 1.2)            # глубокая яма
            - 1.4 * np.exp(-((x + 2.6)**2 + (y + 1.5)**2) / 1.2)  # мелкая яма 1
            - 0.9 * np.exp(-((x - 2.6)**2 + (y + 1.5)**2) / 1.2)  # мелкая яма 2 
            + 0.03 * (x**2 + y**2))                               

def U_vec(v):   # функция для optimize
    return U(v[0], v[1])

# "Скатываем шарик": callback вызывается на каждой итерации minimize
# и записывает текущую точку - получается траектория спуска
def roll_ball(start):
    path = [np.array(start)]
    res = optimize.minimize(U_vec, x0=start, method='Nelder-Mead',
                            callback=lambda v: path.append(v.copy()))
    return np.array(path), res

# Три старта на склонах ОДНОГО холма, чуть с разных сторон вершины
starts = [(0.2, 0.8),
          (-0.8, -0.5), 
          (0.8, -0.5)]

colors = ['red', 'orange', 'blue']

grid = np.linspace(-5, 5, 200)
X, Y = np.meshgrid(grid, grid)

fig = go.Figure()
fig.add_trace(go.Surface(x=X, y=Y, z=U(X, Y),
                         colorscale='RdYlBu_r',   # синий = низко, красный = высоко
                         opacity=0.9, showscale=False))

for start, color in zip(starts, colors):
    path, res = roll_ball(start)
    z_path = U(path[:, 0], path[:, 1]) + 0.06   # чуть приподнять след над поверхностью
    fig.add_trace(go.Scatter3d(x=path[:, 0], y=path[:, 1], z=z_path,
                               mode='lines+markers',
                               line=dict(color=color, width=6),
                               marker=dict(size=3, color=color),
                               name=f'старт {start} → U = {res.fun:.2f}'))
    print(f"старт {start} → минимум ({res.x[0]:5.2f}, {res.x[1]:5.2f}),  "
          f"U = {res.fun:.3f},  шагов: {len(path) - 1}")

fig.update_layout(width=850, height=650,
                  scene=dict(xaxis_title='x', yaxis_title='y', zaxis_title='U(x, y)'),
                  scene_camera=dict(eye=dict(x=1.3, y=-1.5, z=1.1)))
fig.show()

**Ковариационная матрица и расчёт ошибок аппроксимации**

`curve_fit` возвращает два объекта: `popt` - оптимизированные параметры и `pcov` - ковариационную
матрицу. Ковариационная матрица содержит информацию о неопределённости оценок параметров и взаимосвязи между ними. 

Для модели с двумя параметрами $(a, b)$:

$$
\mathbf{C} =
\begin{pmatrix}
\mathrm{Var}(a) & \mathrm{Cov}(a,b) \\
\mathrm{Cov}(a,b) & \mathrm{Var}(b)
\end{pmatrix}
$$

**Диагональ - погрешности параметров.** Диагональные элементы - дисперсии параметров, а
их корни - стандартные ошибки:

$$
\sigma_a = \sqrt{C_{aa}}, \qquad \sigma_b = \sqrt{C_{bb}}
$$

Именно так получают запись результата $a \pm \sigma_a$.

**Недиагональные элементы - ковариации параметров.** Они характеризуют взаимосвязь между оценками параметров. Для удобства эту взаимосвязь часто выражают через коэффициент корреляции:

$$
\rho_{ab} = \frac{\mathrm{Cov}(a,b)}{\sigma_a\,\sigma_b},
\qquad -1 \le \rho_{ab} \le 1
$$

Чем ближе $|\rho_{ab}|$ к 1, тем сильнее параметры коррелируют между собой.

In [ ]:
# Модель: y = a·exp(-x) + b·exp(-1.1x)

def model(x, a, b):
    return a * np.exp(-x) + b * np.exp(-1.1 * x)

a_true, b_true = 1.0, 1.0
np.random.seed(42)
x_data = np.linspace(0, 2, 30)
y_data = model(x_data, a_true, b_true) + np.random.normal(0, 0.01, x_data.size)

popt, pcov = optimize.curve_fit(model, x_data, y_data)
a_fit, b_fit = popt
perr = np.sqrt(np.diag(pcov))
corr = pcov / np.outer(perr, perr)

print("Ковариационная матрица:")
print(pcov)
print("\n")

print(f"a = {a_fit:.2f} ± {perr[0]:.2f}   (истина {a_true})")
print(f"b = {b_fit:.2f} ± {perr[1]:.2f}   (истина {b_true})")
print(f"corr(a, b) = {corr[0, 1]:.5f}")

fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(13, 4))

ax1.plot(x_data, np.exp(-x_data), label='exp(-x)')
ax1.plot(x_data, np.exp(-1.1 * x_data), '--', label='exp(-1.1x)')
ax1.set_title('Слагаемые модели почти совпадают')
ax1.set_xlabel('x')
ax1.legend()
ax1.grid(True)

ax2.plot(x_data, y_data, 'ko', markersize=4, label='Данные')
ax2.plot(x_data, model(x_data, *popt), 'r-', label='curve_fit')
ax2.set_title(f'Идеальная кривая при a = {a_fit:.1f}, b = {b_fit:.1f}')
ax2.set_xlabel('x')
ax2.legend()
ax2.grid(True)

a_values = np.linspace(a_fit - 4*perr[0], a_fit + 4*perr[0], 100)
b_values = np.linspace(b_fit - 4*perr[1], b_fit + 4*perr[1], 100)
rmse_grid = np.zeros((len(b_values), len(a_values)))
for i, b in enumerate(b_values):
    for j, a in enumerate(a_values):
        rmse_grid[i, j] = np.sqrt(np.mean((y_data - model(x_data, a, b))**2))
cs = ax3.contourf(a_values, b_values, rmse_grid, levels=30, cmap='RdYlBu_r')
fig.colorbar(cs, ax=ax3, label='RMSE')

eigval, eigvec = np.linalg.eigh(pcov)
t = np.linspace(0, 2*np.pi, 200)
ax3.plot(a_fit, b_fit, 'ro', markersize=9, label='Минимум')
ax3.plot(a_true, b_true, 'ko', markersize=9, label='Истина')
ax3.set_title(f'corr(a, b) = {corr[0, 1]:.4f}')
ax3.set_xlabel('a')
ax3.set_ylabel('b')
ax3.legend()
ax3.grid(True)

plt.tight_layout()
plt.show()

### 🎲 1.5. Статистика

Результат любого измерения - истинное значение плюс случайная ошибка измерения. При проведении эксперимента повторно результат может отличаться. Подмодуль `scipy.stats` позволяет
описывать этот разброс количественно: распределения, выборочные
характеристики, доверительные интервалы и статистические тесты.

In [ ]:
from scipy import stats

In [ ]:
# 7 параллельных титрований раствора HCl
np.random.seed(42)
c_true = 0.1000                                # истинная концентрация, моль/л
c = c_true + np.random.normal(0, 0.0012, 7)    # 7 измерений со случайной ошибкой
n = len(c)

c_mean = np.mean(c)          # среднее - лучшая оценка истинного значения
s      = np.std(c, ddof=1)   # стандартное отклонение: ddof=1 - деление на n-1
sem    = s / np.sqrt(n)      # стандартная ошибка среднего

print("Измерения, моль/л:", np.round(c, 4))
print(f"Среднее:                  {c_mean:.4f}")
print(f"Разброс одного измерения: s    = {s:.4f}")   # Стандартное отклонение
print(f"Точность среднего:        s/√n = {sem:.4f}") # Разброс среднего из n измерений

In [ ]:
# 200 студентов делают по n титрований каждый;
# точка - среднее одного студента. Чем больше n, тем кучнее средние
plt.figure(figsize=(8, 5))

for n_lab in (3, 10, 30, 100, 300):
    means = np.random.normal(c_true, 0.0012, size=(200, n_lab)).mean(axis=1)
    plt.plot([n_lab] * 200, means, 'o', color='#228CDB', markersize=2, alpha=0.2)

n_grid = np.geomspace(2, 500, 100)
plt.plot(n_grid, c_true + 0.0012 / np.sqrt(n_grid), 'r--', label='истина ± σ/√n')
plt.plot(n_grid, c_true - 0.0012 / np.sqrt(n_grid), 'r--')
plt.axhline(c_true, color='black', ls='--', linewidth=1)
plt.xscale('log')
plt.xlabel('Число титрований n')
plt.ylabel('Среднее одного студента, моль/л')
plt.title('Ошибка среднего падает как 1/√n')
plt.legend()
plt.grid(True)
plt.show()


In [ ]:
# Нормальное распределение - модель случайной ошибки измерения
# Используем выборку титрований из предыдущей ячейки: c_mean, s

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4.5))

# Слева: при большом числе измерений гистограмма превращается в гладкую кривую (PDF)
np.random.seed(42)
c_many = np.random.normal(c_mean, s, 10000)            # 10 000 "титрований" вместо семи
c_grid = np.linspace(c_mean - 4*s, c_mean + 4*s, 400)

ax1.hist(c_many, bins=50, density=True, color='#228CDB', alpha=0.6,
         edgecolor='black', label='10 000 измерений')
ax1.plot(c_grid, stats.norm.pdf(c_grid, c_mean, s), color='#C42021', lw=2,
         label='stats.norm.pdf')
ax1.set_xlabel('c(HCl), моль/л')
ax1.set_ylabel('Плотность вероятности')
ax1.set_title('Гистограмма → гладкая кривая')
ax1.legend()
ax1.grid(True)

# Справа: правило 68-95-99.7 - как читать запись "x ± σ"
mu, sigma = 0, 1
x = np.linspace(-5, 5, 1000)
pdf = stats.norm.pdf(x, mu, sigma)

ax2.plot(x, pdf, color='#C42021', label='PDF')
for k in (1, 2, 3):
    mask = (x >= mu - k*sigma) & (x <= mu + k*sigma)
    share = stats.norm.cdf(k) - stats.norm.cdf(-k)     # доля площади - через CDF
    ax2.fill_between(x[mask], pdf[mask], color='#C42021', alpha=0.45 - 0.14*k,
                     label=f'±{k}σ: {share*100:.1f}%')
ax2.set_xlabel('(x − μ)/σ')
ax2.set_title('Правило 68-95-99.7')
ax2.legend()
ax2.grid(True)

plt.tight_layout()
plt.show()

# Доли площади под кривой считаются через CDF - функцию распределения
for k in (1, 2, 3):
    share = stats.norm.cdf(k) - stats.norm.cdf(-k)
    print(f"P(|x - μ| < {k}σ) = {share:.4f}")

In [ ]:
# Cредние из n равномерно распределённых чисел
# При n = 1 - плоская "полка", при n = 30 - почти идеальная гауссова функция

np.random.seed(42)
fig, axes = plt.subplots(1, 3, figsize=(11, 3.5))

for ax, n_terms in zip(axes, (1, 5, 30)):
    means = np.random.uniform(-1, 1, size=(20000, n_terms)).mean(axis=1)
    ax.hist(means, bins=60, density=True, color='#228CDB', alpha=0.6, edgecolor='none')

    # гауссова кривая с теоретической шириной: σ_среднего = σ_одного / √n
    sigma_n = (2 / np.sqrt(12)) / np.sqrt(n_terms)   # σ равномерного на [-1, 1] = 2/√12
    xs = np.linspace(means.min(), means.max(), 300)
    ax.plot(xs, stats.norm.pdf(xs, 0, sigma_n), 'r-', label='Гаусс с той же σ')
    ax.set_title(f'Среднее из n = {n_terms}')
    ax.set_xlabel('Значение')
    ax.grid(True)

axes[0].set_ylabel('Плотность')
axes[0].legend()
plt.tight_layout()
plt.show()


**Доверительный интервал: что писать в отчёте**

Правило 68-95-99.7 работает, когда $\sigma$ известна точно. Но у нас всего 7 измерений, и вместо $\sigma$ - её оценка $s$, которая сама определена не достоверно. Поэтому в качестве стандартного отклонения берут значения из **распределения Стьюдента**, зависящего от числа степеней свободы $\nu = n - 1$:

$$\bar{x} \pm t_{\alpha,\,\nu} \cdot \frac{s}{\sqrt{n}}$$

Для 95% и $n = 7$: $t = 2.45$ вместо 1.96; при $n = 3$ - уже 4.30; при $n \to \infty$ множитель стремится к гауссову 1.96. Именно в виде $\bar{x} \pm t \cdot s/\sqrt{n}$ результат записывают в отчёте или статье.

Смысл «95%»: если бы 100 студентов повторили наш эксперимент и каждый построил свой интервал, примерно 95 интервалов накрыли бы истинное значение - а 5 промахнулись.


In [ ]:
# Доверительный интервал для среднего: x̄ ± t · s/√n
# Используем те же 7 титрований: c_mean, s, sem, n

alpha = 0.95
t_crit = stats.t.ppf((1 + alpha) / 2, df=n - 1)   # квантиль Стьюдента, ν = n - 1
half = t_crit * sem

print(f"t-множитель (95%, ν = {n-1}): {t_crit:.3f}  (у Гаусса было бы 1.96)")
print(f"Результат: c(HCl) = {c_mean:.4f} ± {half:.4f} моль/л  (P = 95%)")

# То же одной строкой:
ci = stats.t.interval(alpha, df=n - 1, loc=c_mean, scale=sem)
print(f"stats.t.interval:  [{ci[0]:.4f}, {ci[1]:.4f}]")

fig, ax = plt.subplots(figsize=(11, 3.5))

# Слева: во сколько раз t-множитель больше гауссова при маленьких n
n_vals = np.arange(2, 31)
ax.plot(n_vals, stats.t.ppf(0.975, df=n_vals - 1), 'o-', color='#228CDB', label='t (95%)')
ax.axhline(1.96, color='red', ls='--', label='Гаусс: 1.96')
ax.set_xlabel('Число измерений n')
ax.set_ylabel('Множитель')
ax.legend()
ax.grid(True)
plt.show()

**Бутстреп: доверительный интервал без формул**

Настоящих повторных экспериментов у нас нет, но можно «переизмерить» собственную выборку: набрать из неё 7 значений **с возвращением** - одно и то же титрование может попасться дважды, а какое-то ни разу. Повторив это 10 000 раз, получаем 10 000 «виртуальных экспериментов» и смотрим, в каких пределах изменяется их среднее. Для маленьких выборок бутстреп грубее t-интервала, но работает для любой статистики.


In [ ]:
# Бутстреп: 10 000 псевдовыборок из наших 7 титрований

np.random.seed(42)
boot_means = np.array([
    np.mean(np.random.choice(c, size=n, replace=True))   # выборка С ВОЗВРАЩЕНИЕМ
    for _ in range(10_000)
])

print(f"t-интервал:      [{ci[0]:.4f}, {ci[1]:.4f}]   (из предыдущего раздела)")

# В scipy есть и готовая функция:
res = stats.bootstrap((c,), np.mean, confidence_level=0.95, random_state=42)
print(f"stats.bootstrap: [{res.confidence_interval.low:.4f}, "
      f"{res.confidence_interval.high:.4f}]")

lo, hi = np.percentile(boot_means, [2.5, 97.5])

fig, ax = plt.subplots(figsize=(11, 4))
ax.hist(boot_means, bins=60, density=True, color='#2A9D8F', alpha=0.6,
        edgecolor='none', label='Средние 10 000 псевдовыборок')
ax.axvline(lo, color='red', ls='--', label='95% бутстреп-интервал')
ax.axvline(hi, color='red', ls='--')
ax.axvline(c_mean, color='black', label=f'Среднее = {c_mean:.4f}')

# Центральная предельная теорема: среднее распределено ~ нормально с центром c_mean и σ = s/√n
x = np.linspace(boot_means.min(), boot_means.max(), 300)
ax.plot(x, stats.norm.pdf(x, loc=c_mean, scale=sem), 'k--', lw=2,
        label='Гаусс: N(среднее, s/√n)')

ax.set_xlabel('c(HCl), моль/л')
ax.set_ylabel('Плотность')
ax.legend()
ax.grid(True)
plt.show()


**Проверка гипотез: t-тесты**

1. Совпадает ли результат с значением для стандартного образца? Проверяем используя одновыборочный t-тест, `stats.ttest_1samp`
2. Дают ли два метода одинаковый результат? Проверяем используя двухвыборочный t-тест, `stats.ttest_ind`

Оба возвращают **p-value** - вероятность получить наблюдаемое (или ещё большее) расхождение случайно, если на самом деле различия нет. Малое значение p (обычно < 0.05) - расхождение вряд ли случайно, различие **значимо**.

⚠️ Большое значение p **не доказывает равенство** - он означает, что данных недостаточно, чтобы увидеть различие.

In [ ]:
# Одновыборочный t-тест: сравнение со стандартным образцом
# Аттестованное значение c_атт = 0.1005 моль/л. Есть ли у методики систематическая ошибка?

c_cert = 0.1005   # аттестованное значение, моль/л

t_stat, p_value = stats.ttest_1samp(c, popmean=c_cert)
print(f"Наш результат:  {c_mean:.4f} моль/л  (7 титрований)")
print(f"Аттестовано:    {c_cert:.4f} моль/л")
print(f"t = {t_stat:.2f}, p = {p_value:.3f}")

if p_value < 0.05:
    print("p < 0.05: расхождение значимо - у методики есть систематическая ошибка")
else:
    print("p ≥ 0.05: расхождение объяснимо случайным разбросом")


In [ ]:
# Двухвыборочный t-тест: дают ли два метода одинаковый результат?
# У метода Б мы задаём систематическое смещение +0.0010 моль/л.
# Посмотрим, увидит ли его тест при 8 измерениях и при 50

bias = 0.0010
np.random.seed(1)

c_A8  = np.random.normal(c_true,        0.0012, 8)
c_B8  = np.random.normal(c_true + bias, 0.0015, 8)
c_A50 = np.random.normal(c_true,        0.0012, 50)
c_B50 = np.random.normal(c_true + bias, 0.0015, 50)

t8,  p8  = stats.ttest_ind(c_A8,  c_B8)
t50, p50 = stats.ttest_ind(c_A50, c_B50)

print(f"n = 8:  метод А {c_A8.mean():.4f},  метод Б {c_B8.mean():.4f},  p = {p8:.3f}")
print(f"n = 50: метод А {c_A50.mean():.4f}, метод Б {c_B50.mean():.4f}, p = {p50:.5f}")
print(f"\nСмещение +{bias} моль/л реально существует, но при n = 8 тест его не видит:")
print("большой p означает «данных мало», а не «методы одинаковы»")

fig, axes = plt.subplots(1, 2, figsize=(11, 4), sharey=True)
np.random.seed(0)   # джиттер точек по горизонтали, чтобы не слипались
for ax, cA, cB, p in [(axes[0], c_A8, c_B8, p8), (axes[1], c_A50, c_B50, p50)]:
    for pos, data, color in [(1, cA, '#228CDB'), (2, cB, '#E76F51')]:
        jitter = np.random.uniform(-0.08, 0.08, len(data))
        ax.plot(pos + jitter, data, 'o', color=color, alpha=0.5)
        ax.hlines(data.mean(), pos - 0.2, pos + 0.2, color=color, lw=3)
    ax.axhline(c_true, color='gray', ls='--', lw=1)
    ax.set_xticks([1, 2], ['Метод А', 'Метод Б'])
    ax.set_xlim(0.5, 2.5)
    ax.set_title(f'n = {len(cA)}:  p = {p:.3f}')
    ax.grid(True, axis='y')
axes[0].set_ylabel('c(HCl), моль/л')
plt.tight_layout()
plt.show()

### 🧪 1.6 Химические задачи 

In [ ]:
# Поиск корня одного уравнения: pH слабой кислоты 
Ka = 1.8e-5   # константа диссоциации уксусной кислоты
C0 = 0.1      # начальная концентрация, моль/л

def equation(x):                 # x = [H+]
    return x**2 + Ka*x - Ka*C0   # из материального баланса и выражения для Ka

h = optimize.brentq(equation, 1e-14, 1.0)   # корень между a=1e-14 и b=1.0
print(f"[H+] = {h:.3e} моль/л, pH = {-np.log10(h):.2f}")

# Сравним с приближённой формулой (справедлива при малой степени диссоциации)
h_approx = np.sqrt(Ka * C0)
print(f"Приближение √(Ka·C0): pH = {-np.log10(h_approx):.2f}")

In [ ]:
# Расчет равновесных параметров реакционной смеси

# Синтез аммиака: N2 + 3H2 ⇌ 2NH3, Kp при 673 K ≈ 1.6e-4 атм⁻²
# Загружаем 1 моль N2 и 3 моль H2, общее давление P. x — степень превращения N2.
# Мольные количества: N2: 1−x, H2: 3(1−x), NH3: 2x, всего: 4−2x

Kp = 1.6e-4   # атм^-2
P  = 300      # атм (промышленные условия)

def equilibrium(x):
    
    n_tot = 4 - 2*x
    p_N2  = (1-x)/n_tot*P
    p_H2  = 3*(1-x)/n_tot*P
    p_NH3 = 2*x/n_tot*P

    return p_NH3**2 / (p_N2 * p_H2**3) - Kp

x_eq = optimize.brentq(equilibrium, 1e-6, 1 - 1e-6)
print(f"Степень превращения N2: {x_eq:.3f}")
print(f"Мольная доля NH3 в смеси: {2*x_eq/(4-2*x_eq):.3f}")


In [ ]:
# Система одновременных равновесий
# Паровая конверсия метана при 1000 K, P = 1 атм:
#   (1) CH4 + H2O ⇌ CO + 3H2     K1 ≈ 27
#   (2) CO  + H2O ⇌ CO2 + H2     K2 ≈ 1.4
# Рассчитать равновесный состав смеси

K1, K2, P = 27.0, 1.4, 1.0

def equations(v):
    x, y = v
    n_tot = 3 + 2*x # Суммарное количество моль от глубины реакции x 

    # Парциальные давления по закону Дальтона: p_i = (n_i / n_общ)·P
    p_CH4 = (1 - x)     / n_tot * P
    p_H2O = (2 - x - y) / n_tot * P
    p_CO  = (x - y)     / n_tot * P
    p_H2  = (3*x + y)   / n_tot * P
    p_CO2 = y           / n_tot * P

    eq1 = p_CO  * p_H2**3 - K1 * p_CH4 * p_H2O
    eq2 = p_CO2 * p_H2    - K2 * p_CO  * p_H2O
    return [eq1, eq2]

x, y = optimize.fsolve(equations, [0.9, 0.3])   # старт из физичной области

print(f"Конверсия CH4: x = {x:.3f},  глубина реакции сдвига: y = {y:.3f}\n")
print("Равновесный состав смеси:")
n_tot = 3 + 2*x
for name, n_i in [('CH4', 1-x), ('H2O', 2-x-y), ('CO', x-y),
                  ('H2', 3*x+y), ('CO2', y)]:
    print(f"  {name:>4}: {n_i:.3f} моль  ({n_i/n_tot:.1%})")

In [ ]:
# Необходимо рассчитать работу, которую совершает 1 моль углекислого газа при изотермическом расширении, подчиняясь уравнению Ван-дер-Ваальса.
# Параметры Ван-дер-Ваальса для CO2
a, b, R = 3.59, 0.0427, 0.08206 # л^2*атм/моль^2, л/моль, л*атм/(моль*К)
T = 300.0 # К

# Функция давления от объема
def P_vdw(V):
    return (R * T) / (V - b) - a / V**2

V_start = 0.5  # л/моль
V_end = 2.0    # л/моль

# quad возвращает само значение интеграла и оценку абсолютной ошибки
work, error = integrate.quad(P_vdw, V_start, V_end)

# Переводим из (л*атм) в Джоули (1 л*атм = 101.325 Дж)
work_J = work * 101.325
print(f"Работа расширения: {work_J:.2f} Дж (Погрешность вычисления: {error:.2e})")

V_plot = np.linspace(0.3, 2.5, 200)
P_plot = P_vdw(V_plot)

plt.figure(figsize=(8, 5))
plt.plot(V_plot, P_plot, 'b-', linewidth=2, label='Изотерма CO$_2$ (300 К)')

V_fill = np.linspace(V_start, V_end, 100)
plt.fill_between(V_fill, P_vdw(V_fill), color='skyblue', alpha=0.5, 
                 label=f'Работа $W$ = {work_J:.0f} Дж')

plt.xlabel('Молярный объем $V$, л/моль')
plt.ylabel('Давление $P$, атм')
plt.title('Работа изотермического расширения реального газа')
plt.legend()
plt.grid(True, linestyle=':')
plt.show()

In [ ]:
# Разложение пероксида водорода при фотохимическом инициировании протекает
# по цепному неразветвлённому механизму:
#
#   (0) H2O2        -> 2 OH•             k0 = 1e-7 М·с⁻¹
#   (1) OH•  + H2O2 -> HO2• + H2O        k1 = 3e7  М⁻¹·с⁻¹
#   (2) HO2• + H2O2 -> H2O + O2 + OH•    k2 = 30   М⁻¹·с⁻¹
#   (3) HO2• + HO2• -> H2O2 + O2         k3 = 2e5  М⁻¹·с⁻¹
#
# Задача: найти зависимость концентраций всех веществ от времени
# при начальной концентрации перекиси [H2O2]₀ = 0.1 М.

k0, k1, k2, k3 = 1e-7, 3e7, 30, 2e5
C0 = [0.1, 0, 0, 0, 0]                    # [H2O2, OH•, HO2•, H2O, O2], моль/л

def rhs(t, y, k0, k1, k2, k3):
    H2O2, OH, HO2, H2O, O2 = y

    dH2O2_dt = -k0 - k1*OH*H2O2 - k2*HO2*H2O2 + k3*HO2**2
    dOH_dt   = 2*k0 - k1*OH*H2O2 + k2*HO2*H2O2
    dHO2_dt  =        k1*OH*H2O2 - k2*HO2*H2O2 - 2*k3*HO2**2
    dH2O_dt  =        k1*OH*H2O2 + k2*HO2*H2O2
    dO2_dt   =                     k2*HO2*H2O2 + k3*HO2**2

    return [dH2O2_dt, dOH_dt, dHO2_dt, dH2O_dt, dO2_dt]

t_end = 20  # с
t = np.linspace(0, t_end, 500)
sol = integrate.solve_ivp(rhs,
                          (0, t_end),
                          C0,
                          t_eval=t,
                          args=(k0, k1, k2, k3),
                          method='Radau',   # Стандартный RK45 здесь не подойдет 
                          rtol=1e-8,        # относительная точность: ~8 верных значащих цифр
                          atol=1e-15        # абсолютный порог: ниже него концентрация считается
                                            # "нулём"; должен быть меньше [OH•] ~ 1e-12 М,
                                            # иначе решатель не станет разрешать радикалы
)

H2O2, OH, HO2, H2O, O2 = sol.y

for c, name, col in [(H2O2, 'H$_2$O$_2$', 'blue'),
                     (OH,   'OH•',        'purple'),
                     (HO2,  'HO$_2$•',    'orange'),
                     (H2O,  'H$_2$O',     'green'),
                     (O2,   'O$_2$',      'red')]:
    plt.figure(figsize=(7, 4))
    plt.plot(t, c, color=col)
    plt.xlabel('t, с'); plt.ylabel('концентрация, моль/л')
    plt.title(name)
    plt.grid(alpha=0.3)
    plt.show()


In [ ]:
# Последовательный гидролиз диэфира (в избытке воды — псевдопервый порядок):
#
#   (1) D -> M    k1      диэфир -> моноэфир
#   (2) M -> P    k2      моноэфир -> кислота
#
# Задача: по экспериментальным зависимостям концентраций от времени
# найти константы скорости k1 и k2 (мин⁻¹) и их погрешности.
# Начальная концентрация диэфира [D]₀ = 0.100 М.

C0 = [0.100, 0.0, 0.0]        # [D, M, P], моль/л

# Экспериментальные данные: время (мин) и концентрации (моль/л)
df = pd.read_csv('Hydrolysis.csv')
print(df.head())

t_exp = df['t_min'].to_numpy(float)
D_exp = df['D'].to_numpy()
M_exp = df['M'].to_numpy()
P_exp = df['P'].to_numpy()

def rhs(t, y, k1, k2):
    D, M, P = y

    dD_dt = -k1*D           # диэфир только расходуется в (1)
    dM_dt =  k1*D - k2*M    # моноэфир: приход из (1), расход в (2)
    dP_dt =         k2*M    # продукт только накапливается в (2)

    return [dD_dt, dM_dt, dP_dt]

# Модель для curve_fit: при каждом вызове решает ОДУ с пробными k1, k2
# и возвращает три кривые, склеенные в один вектор (как и ydata ниже)

def model(t, k1, k2):
    s = integrate.solve_ivp(rhs, (0, t.max()), C0, t_eval=t, args=(k1, k2), rtol=1e-8)
    return np.concatenate([s.y[0], s.y[1], s.y[2]])

popt, pcov = optimize.curve_fit(model, t_exp,
                                np.concatenate([D_exp, M_exp, P_exp]),
                                p0=[0.1, 0.01],        # стартовое приближение
                                bounds=(0, np.inf))    # константы неотрицательны
perr = np.sqrt(np.diag(pcov))      # стандартные погрешности из ковариационной матрицы

print(f"k1 = {popt[0]:.3f} ± {perr[0]:.3f} мин⁻¹")
print(f"k2 = {popt[1]:.3f} ± {perr[1]:.3f} мин⁻¹")

# График: точки — «эксперимент», линии - модель с подобранными константами
t_fine = np.linspace(0, t_exp.max(), 300)
sol = integrate.solve_ivp(rhs, (0, t_exp.max()), C0, t_eval=t_fine, args=(*popt,), rtol=1e-8)

plt.figure(figsize=(8, 5))
for y_e, y_m, name, col in [(D_exp, sol.y[0], 'диэфир D',   'blue'),
                            (M_exp, sol.y[1], 'моноэфир M', 'red'),
                            (P_exp, sol.y[2], 'кислота P',  'green')]:
    plt.plot(t_exp, y_e, 'o', color=col, label=f'{name} (эксп.)')
    plt.plot(t_fine, y_m, '-', color=col, alpha=0.7, label=f'{name} (модель)')
plt.xlabel('t, мин'); plt.ylabel('концентрация, моль/л')
plt.legend(); plt.grid(alpha=0.3)
plt.show()


# 💻 Глава 2. Способы компьютерного представления молекул

**Понятия о топологии и координатах**

Молекулу можно описать на двух уровнях:

- **Топология** - содержит информацию о структуре, атомах есть и связях и не содержит информацию о положении атомов в пространстве.
- **Геометрия** - определяет положение каждого атома в пространстве: например тройка чисел $(x, y, z)$.

У одной топологии может быть несколько геометрий: этанол `CCO` - это одна молекула, но каждый поворот вокруг связи C-C даёт новый набор координат (конформер). Обратное тоже верно: набор координат атомов может быть отнесён к разным топологиям, но обычно топологию можно определить по межатомным расстояниям.

In [ ]:
from rdkit import Chem
from rdkit.Chem import Draw, AllChem

### 🧬 2.1 SMILES - строчная запись молекулы

**SMILES** (Simplified Molecular Input Line Entry System) - способ записать топологию молекулы одной строкой текста. Именно в таком виде молекулы обычно хранятся в базах данных.

| Элемент записи | Пример | Значение |
|---|---|---|
| Атомы | `C`, `N`, `O`, `Cl` | водороды дописываются автоматически по валентности |
| Одинарная связь | `CC` | этан (символ `-` можно опускать) |
| Двойная / тройная связь | `C=C`, `C#C` | этилен, ацетилен |
| Ветвление | `CC(C)C` | скобки - боковая цепь (изобутан) |
| Цикл | `C1CCCCC1` | цифра помечает атомы замыкания цикла (циклогексан) |
| Ароматика | `c1ccccc1` | строчные буквы - ароматические атомы (бензол) |
| Заряд | `[NH4+]`, `[O-]` | атомы с зарядом - в квадратных скобках |

Ограничения:

- SMILES хранит только топологию - никакой информации о 3D-геометрии и конформациях
- запись не уникальна: одна молекула - много вариантов строк (может быть исправлено канонизацией)
- таутомеры формально считаются разными молекулами
- плохо подходит для неорганики: координационные и металлоорганические соединения, многоцентровые связи

In [ ]:
# Chem.MolFromSmiles превращает строку в объект Mol - молекулярный граф
smiles = {
    'метан':             'C',
    'этанол':           'CCO',
    'уксусная кислота': 'CC(=O)O',
    'бензол':           'c1ccccc1',
    'аспирин':          'CC(=O)Oc1ccccc1C(=O)O',
    'кофеин':           'Cn1cnc2c1c(=O)n(C)c(=O)n2C',
    'парацетамол':      'CC(=O)Nc1ccc(O)cc1',
    'ибупрофен':        'CC(C)Cc1ccc(cc1)C(C)C(=O)O',
}

mols = [Chem.MolFromSmiles(s) for s in smiles.values()]
Draw.MolsToGridImage(mols, molsPerRow=4, subImgSize=(220, 180), legends=list(smiles.keys()))

In [ ]:
# Один и тот же этанол можно записать по-разному - SMILES не уникален
variants = ['CCO', 'OCC', 'C(O)C', '[CH3][CH2][OH]']

for s in variants:
    print(f"{s:16s} -> {Chem.MolToSmiles(Chem.MolFromSmiles(s))}")

# MolToSmiles приводит запись к КАНОНИЧЕСКОЙ форме - она уже уникальна.
# Так убирают дубликаты в базах данных

In [ ]:
mol = Chem.MolFromSmiles('CC(=O)O')   # уксусная кислота

print(f"Атомов (без H): {mol.GetNumAtoms()}, связей: {mol.GetNumBonds()}\n")

print("Атомы:")
for atom in mol.GetAtoms():
    print(f"  {atom.GetIdx()}: {atom.GetSymbol():2s}  связей: {atom.GetDegree()},  "
          f"неявных H: {atom.GetTotalNumHs()}")

print("\nСвязи:")
for bond in mol.GetBonds():
    print(f"  {bond.GetBeginAtom().GetSymbol()}{bond.GetBeginAtomIdx()} - "
          f"{bond.GetEndAtom().GetSymbol()}{bond.GetEndAtomIdx()}: {bond.GetBondType()}")

# Явные водороды добавляются отдельной операцией
mol_H = Chem.AddHs(mol)
print(f"\nПосле AddHs: атомов {mol_H.GetNumAtoms()} - брутто-формула C2H4O2")


### 🗂️ 2.2 Другие представления: InChI, Molfile, PDB, XYZ

| Формат | Что хранит | Где используется |
|---|---|---|
| SMILES | граф, одна строка | базы данных, ввод структур |
| InChI / InChIKey | граф, каноническая строка / её хеш | поиск в PubChem и Google |
| Molfile (.mol, .sdf) | граф + 2D/3D координаты | ChemDraw, базы структур |
| PDB | 3D-координаты + цепи, остатки, метаданные | белки и биомолекулы, Protein Data Bank |
| XYZ | **только** элементы и координаты | квантовая химия (ORCA, Gaussian) |

In [ ]:
# Конвертация одного формата в другой 

mol = Chem.MolFromSmiles('CC(=O)Oc1ccccc1C(=O)O')

# Граф -> канонические строки
print("SMILES:  ", Chem.MolToSmiles(mol))
print("InChI:   ", Chem.MolToInchi(mol))
print("InChIKey:", Chem.MolToInchiKey(mol))   # хеш фиксированной длины - удобно гуглить

In [ ]:
# Граф -> Molfile: нужны координаты, сначала вычисляем 2D
AllChem.Compute2DCoords(mol)
print(Chem.MolToMolBlock(mol))

In [ ]:
from urllib.request import urlopen

pdb_id = '1YJP'
pdb_text = urlopen(f'https://files.rcsb.org/download/{pdb_id}.pdb').read().decode()

# Печатаем только содержательные записи (служебные REMARK пропускаем)
for line in pdb_text.splitlines():
    if line.startswith(('HEADER', 'TITLE', 'EXPDTA', 'SEQRES', 'CRYST1','ATOM', 'TER', 'HETATM', 'END')):
        print(line)

In [ ]:
# Граф -> 3D: добавить H -> сгенерировать конформер -> оптимизировать силовым полем
mol3d = Chem.AddHs(mol)
AllChem.EmbedMolecule(mol3d, randomSeed=42)   # Генерация стартовых 3D-координат
AllChem.MMFFOptimizeMolecule(mol3d)           # Оптимизация геометрии (MMFF94) 

# XYZ - стандартный формат для квантово-химических программ (ORCA, Gaussian)
print(Chem.MolToXYZBlock(mol3d))

**Z-матрица: те же координаты, но точкой отсчёта считается первый атом молекулы**

XYZ задаёт положение атома тройкой $(x, y, z)$ относительно внешних осей, которые для молекулы ничего не значат. **Z-матрица** задаёт ту же геометрию через **внутренние координаты**: каждый атом определяется относительно уже введённых - длиной связи до одного, валентным углом с двумя и торсионным углом с тремя.

- Первый атом - без координат, второй - только расстояние, третий - расстояние и угол, дальше - по три числа на атом. Итого $3N - 6$ чисел - ровно столько у молекулы степеней свободы (в XYZ чисел $3N$: лишние шесть - это перенос и поворот молекулы целиком).
- Главное удобство: «повернуть группу вокруг связи» можно изменив **одно число** (Например торсионный угол).

In [ ]:
from rdkit.Chem import rdMolTransforms

mol = Chem.MolFromSmiles('CC')
mol = Chem.AddHs(mol)
AllChem.EmbedMolecule(mol, randomSeed=42)
AllChem.MMFFOptimizeMolecule(mol)

def z_matrix(mol):
    conf = mol.GetConformer()
    lines = ['подпись:  атом  R, Å   атом  угол°   атом  торсион°']
    for i, atom in enumerate(mol.GetAtoms()):
        line = f"{atom.GetSymbol():2s}"
        if i >= 1:   # j - связанный с i атом из уже определённых
            j = min(a.GetIdx() for a in atom.GetNeighbors() if a.GetIdx() < i)
            line += f"  {j+1:2d} {rdMolTransforms.GetBondLength(conf, i, j):7.4f}"
        if i >= 2:   # k - уже определённый сосед атома j
            k = min(a.GetIdx() for a in mol.GetAtomWithIdx(j).GetNeighbors()
                    if a.GetIdx() < i and a.GetIdx() != i)
            line += f"  {k+1:2d} {rdMolTransforms.GetAngleDeg(conf, i, j, k):7.2f}"
        if i >= 3:   # l - уже определённый сосед атома k
            cand = [a.GetIdx() for a in mol.GetAtomWithIdx(k).GetNeighbors()
                    if a.GetIdx() < i and a.GetIdx() not in (i, j)]
            l = min(cand) if cand else min(x for x in range(i) if x not in (j, k))
            line += f"  {l+1:2d} {rdMolTransforms.GetDihedralDeg(conf, i, j, k, l):8.2f}"
        lines.append(line)
    return "\n".join(lines)

print(z_matrix(mol))